# Hospital Readmission Risk Analysis — Data Understanding

This notebook profiles the raw hospital encounter dataset without cleaning, transforming, or modifying it. The checks establish the dataset's structure, identifiers, target categories, missing-value patterns, and duplicate counts before later analysis decisions are made.

## 1. Imports

Only the libraries needed to locate and inspect the dataset are imported. Keeping imports minimal makes the notebook easier to understand and reproduce.

In [2]:
from pathlib import Path

import pandas as pd

## 2. Load Dataset

The code locates the repository whether Jupyter starts in the project root or in the `notebooks` directory. The raw CSV is read into `df` and is not written back or modified.

In [3]:
current_directory = Path.cwd().resolve()
candidate_roots = [current_directory, *current_directory.parents]
project_root = next(
    (root for root in candidate_roots if (root / "data/raw/diabetic_data.csv").is_file()),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not find data/raw/diabetic_data.csv from the current directory "
        "or any of its parent directories."
    )

data_path = project_root / "data" / "raw" / "diabetic_data.csv"
df = pd.read_csv(data_path)

print(f"Dataset loaded from: {data_path}")

Dataset loaded from: /Users/josmithareddy/Documents/Data-Analyst-Projects/hospital-readmission-risk-analysis/data/raw/diabetic_data.csv


## 3. Dataset Dimensions

Row and column counts establish the dataset's basic size and help analysts judge the scope of later validation and analysis work.

In [4]:
number_of_rows, number_of_columns = df.shape

print(f"Number of rows: {number_of_rows:,}")
print(f"Number of columns: {number_of_columns:,}")

Number of rows: 101,766
Number of columns: 50


## 4. Preview Data

Viewing records from the beginning, end, and a reproducible random sample helps reveal the layout and variety of values without drawing conclusions about the full dataset.

In [6]:
print("First 5 rows:")
print (df.head(5))

print("Last 5 rows:")
print (df.tail(5))

print("Random sample of 5 rows:")
print (df.sample(n=min(5, len(df)), random_state=42))

First 5 rows:
   encounter_id  patient_nbr             race  gender      age weight  \
0       2278392      8222157        Caucasian  Female   [0-10)      ?   
1        149190     55629189        Caucasian  Female  [10-20)      ?   
2         64410     86047875  AfricanAmerican  Female  [20-30)      ?   
3        500364     82442376        Caucasian    Male  [30-40)      ?   
4         16680     42519267        Caucasian    Male  [40-50)      ?   

   admission_type_id  discharge_disposition_id  admission_source_id  \
0                  6                        25                    1   
1                  1                         1                    7   
2                  1                         1                    7   
3                  1                         1                    7   
4                  1                         1                    7   

   time_in_hospital  ... citoglipton insulin  glyburide-metformin  \
0                 1  ...          No      No       

## 5. Column Information

Column names, inferred data types, and non-null counts provide an initial schema review and identify fields that may need closer investigation later.

In [7]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.rename("data_type").to_frame())

print("DataFrame information:")
df.info()

Column names:
['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']

Data types:


,data_type
encounter_id,int64
patient_nbr,int64
race,str
gender,str
age,str
weight,str
admission_type_id,int64
discharge_disposition_id,int64
admission_source_id,int64
time_in_hospital,int64


DataFrame information:
<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   encounter_id              101766 non-null  int64
 1   patient_nbr               101766 non-null  int64
 2   race                      101766 non-null  str  
 3   gender                    101766 non-null  str  
 4   age                       101766 non-null  str  
 5   weight                    101766 non-null  str  
 6   admission_type_id         101766 non-null  int64
 7   discharge_disposition_id  101766 non-null  int64
 8   admission_source_id       101766 non-null  int64
 9   time_in_hospital          101766 non-null  int64
 10  payer_code                101766 non-null  str  
 11  medical_specialty         101766 non-null  str  
 12  num_lab_procedures        101766 non-null  int64
 13  num_procedures            101766 non-null  int64
 14  num_medi

## 6. Unique Patient and Encounter Checks

Identifier checks distinguish encounter-level records from unique patients and test whether the expected encounter key uniquely identifies each row. Columns are validated before calculations are attempted.

In [8]:
encounter_id_column = "encounter_id"
patient_id_column = "patient_nbr"

print(f"Total encounter rows: {len(df):,}")

if encounter_id_column in df.columns:
    unique_encounter_ids = df[encounter_id_column].nunique(dropna=True)
    encounter_ids_are_unique = df[encounter_id_column].is_unique
    print(f"Unique encounter IDs: {unique_encounter_ids:,}")
    print(f"Encounter IDs are unique: {encounter_ids_are_unique}")
else:
    unique_encounter_ids = None
    encounter_ids_are_unique = None
    print(f"Column not found: {encounter_id_column}")

if patient_id_column in df.columns:
    unique_patient_numbers = df[patient_id_column].nunique(dropna=True)
    print(f"Unique patient numbers: {unique_patient_numbers:,}")
else:
    unique_patient_numbers = None
    print(f"Column not found: {patient_id_column}")

Total encounter rows: 101,766
Unique encounter IDs: 101,766
Encounter IDs are unique: True
Unique patient numbers: 71,518


## 7. Readmission Target Exploration

Reviewing the original target categories and their frequencies establishes how readmission is represented. The target remains in its source form; no binary flag is created.

In [9]:
target_column = "readmitted"

if target_column in df.columns:
    readmission_unique_values = df[target_column].unique()
    readmission_counts = df[target_column].value_counts(dropna=False)
    readmission_percentages = (
        df[target_column].value_counts(dropna=False, normalize=True).mul(100)
    )
    readmission_distribution = pd.DataFrame(
        {
            "count": readmission_counts,
            "percentage": readmission_percentages,
        }
    )

    print("Unique readmission values:")
    print(readmission_unique_values)
    print("\nReadmission category distribution:")
    display(readmission_distribution)
else:
    readmission_unique_values = None
    readmission_distribution = None
    print(f"Column not found: {target_column}")

Unique readmission values:
<ArrowStringArray>
['NO', '>30', '<30']
Length: 3, dtype: str

Readmission category distribution:


,count,percentage
readmitted,,
NO,54864,53.911916
>30,35545,34.928169
<30,11357,11.159916


## 8. Missing Value Investigation

Standard null values and literal `?` placeholders are measured separately because pandas does not treat `?` as missing by default. The placeholders are counted but are not replaced.

In [11]:
null_counts = df.isna().sum()
null_summary = pd.DataFrame(
    {
        "null_count": null_counts,
        "null_percentage": null_counts.div(len(df)).mul(100),
    }
).query("null_count > 0").sort_values("null_percentage", ascending=False)

print("Columns containing standard null/NaN values:")
if null_summary.empty:
    print("No standard null/NaN values found.")
else:
    print (null_summary)

question_mark_counts = df.eq("?").sum()
question_mark_summary = pd.DataFrame(
    {
        "question_mark_count": question_mark_counts,
        "question_mark_percentage": question_mark_counts.div(len(df)).mul(100),
    }
).query("question_mark_count > 0").sort_values(
    "question_mark_percentage", ascending=False
)

print("\nColumns containing '?' placeholders:")
if question_mark_summary.empty:
    print("No '?' placeholders found.")
else:
    print (question_mark_summary)

Columns containing standard null/NaN values:
               null_count  null_percentage
max_glu_serum       96420        94.746772
A1Cresult           84748        83.277322

Columns containing '?' placeholders:
                   question_mark_count  question_mark_percentage
weight                           98569                 96.858479
medical_specialty                49949                 49.082208
payer_code                       40256                 39.557416
race                              2273                  2.233555
diag_3                            1423                  1.398306
diag_2                             358                  0.351787
diag_1                              21                  0.020636


## 9. Duplicate Investigation

Duplicate checks identify repeated complete records and repeated encounter identifiers. Nothing is removed because duplication must be understood before any cleaning decision is made.

In [12]:
full_duplicate_row_count = df.duplicated().sum()
print(f"Fully duplicated rows: {full_duplicate_row_count:,}")

if encounter_id_column in df.columns:
    duplicate_encounter_id_count = df[encounter_id_column].duplicated().sum()
    duplicate_encounter_ids = (
        df.loc[
            df[encounter_id_column].duplicated(keep=False), encounter_id_column
        ]
        .value_counts(dropna=False)
        .rename("row_count")
        .to_frame()
    )
    print(f"Rows with an encounter ID duplicated after its first occurrence: {duplicate_encounter_id_count:,}")
    if duplicate_encounter_ids.empty:
        print("No duplicate encounter IDs found.")
    else:
        display(duplicate_encounter_ids)
else:
    duplicate_encounter_id_count = None
    duplicate_encounter_ids = None
    print(f"Duplicate encounter ID check skipped; column not found: {encounter_id_column}")

Fully duplicated rows: 0
Rows with an encounter ID duplicated after its first occurrence: 0
No duplicate encounter IDs found.


## 10. Initial Data Profiling Summary

This programmatically generated snapshot consolidates the main profiling measures for review. It reports observed values only and does not make cleaning or modeling decisions.

In [14]:
profiling_summary = pd.Series(
    {
        "rows": len(df),
        "columns": df.shape[1],
        "unique_patients": unique_patient_numbers,
        "columns_containing_question_mark": len(question_mark_summary),
        "full_duplicate_rows": full_duplicate_row_count,
    },
    name="value",
)

print("Initial data profiling summary:")
print (profiling_summary.to_frame())

print("Readmission category distribution:")
if readmission_distribution is not None:
    print (readmission_distribution)
else:
    print(f"Unavailable; column not found: {target_column}")

Initial data profiling summary:
                                   value
rows                              101766
columns                               50
unique_patients                    71518
columns_containing_question_mark       7
full_duplicate_rows                    0
Readmission category distribution:
            count  percentage
readmitted                   
NO          54864   53.911916
>30         35545   34.928169
<30         11357   11.159916


In [15]:
patient_encounter_counts = (
    df.groupby("patient_nbr")
    .size()
    .sort_values(ascending=False)
)

print("Encounter count for each patient:")
print(patient_encounter_counts.head(20))

Encounter count for each patient:
patient_nbr
88785891    40
43140906    28
1660293     23
23199021    23
88227540    23
23643405    22
84428613    22
92709351    21
23398488    20
29903877    20
90609804    20
37096866    20
88789707    20
89472402    20
88681950    19
88479036    19
97391007    19
3401055     18
91751121    18
84348792    18
dtype: int64


In [17]:
encounter_frequency = pd.Series({
    "1 encounter": (patient_encounter_counts == 1).sum(),
    "2 encounters": (patient_encounter_counts == 2).sum(),
    "3 encounters": (patient_encounter_counts == 3).sum(),
    "4 encounters": (patient_encounter_counts == 4).sum(),
    "5+ encounters": (patient_encounter_counts >= 5).sum()
})

encounter_frequency_percentage = (
        encounter_frequency / len(patient_encounter_counts) * 100
)

patient_frequency_summary = pd.DataFrame({
    "patients": encounter_frequency,
    "percentage": encounter_frequency_percentage
})

print(patient_frequency_summary)
repeat_patients = (patient_encounter_counts > 1).sum()
one_time_patients = (patient_encounter_counts == 1).sum()

repeat_patient_percentage = (
        repeat_patients / len(patient_encounter_counts) * 100
)

print("\nUnique patients:", len(patient_encounter_counts))
print("One-time patients:", one_time_patients)
print("Repeat patients:", repeat_patients)
print(f"Repeat patient percentage: {repeat_patient_percentage:.2f}%")

               patients  percentage
1 encounter       54745   76.547163
2 encounters      10434   14.589334
3 encounters       3328    4.653374
4 encounters       1421    1.986912
5+ encounters      1590    2.223217

Unique patients: 71518
One-time patients: 54745
Repeat patients: 16773
Repeat patient percentage: 23.45%


In [18]:
repeat_patient_ids = patient_encounter_counts[
    patient_encounter_counts > 1
    ].index

df_repeat = df[
    df["patient_nbr"].isin(repeat_patient_ids)
].copy()

print("Repeat patients:", len(repeat_patient_ids))
print("Encounters belonging to repeat patients:", len(df_repeat))
print("\nFirst 5 repeat-patient encounters:")
print(df_repeat.head())

Repeat patients: 16773
Encounters belonging to repeat patients: 47021

First 5 repeat-patient encounters:
    encounter_id  patient_nbr       race  gender      age weight  \
12         40926     85504905  Caucasian  Female  [40-50)      ?   
27        248916    115196778  Caucasian  Female  [50-60)      ?   
28        250872     41606064  Caucasian    Male  [20-30)      ?   
32        260166     80845353  Caucasian  Female  [70-80)      ?   
33        293058    114715242  Caucasian    Male  [60-70)      ?   

    admission_type_id  discharge_disposition_id  admission_source_id  \
12                  1                         3                    7   
27                  1                         1                    1   
28                  2                         1                    2   
32                  1                         1                    7   
33                  2                         6                    2   

    time_in_hospital  ... citoglipton insulin  glybu

In [19]:
one_time_patient_ids = patient_encounter_counts[
    patient_encounter_counts == 1
    ].index

df_one_time = df[
    df["patient_nbr"].isin(one_time_patient_ids)
].copy()

print("One-time patients:", len(one_time_patient_ids))
print("Encounters belonging to one-time patients:", len(df_one_time))

One-time patients: 54745
Encounters belonging to one-time patients: 54745


In [20]:
comparison_columns = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

one_time_averages = df_one_time[comparison_columns].mean()

repeat_averages = df_repeat[comparison_columns].mean()

comparison = pd.DataFrame({
    "one_time_patients": one_time_averages,
    "repeat_patients": repeat_averages
})

comparison["difference"] = (
        comparison["repeat_patients"]
        - comparison["one_time_patients"]
)

print(comparison.round(2))

                    one_time_patients  repeat_patients  difference
time_in_hospital                 4.22             4.60        0.38
num_lab_procedures              43.38            42.76       -0.62
num_procedures                   1.47             1.19       -0.27
num_medications                 15.63            16.48        0.85
number_outpatient                0.27             0.49        0.22
number_emergency                 0.09             0.32        0.23
number_inpatient                 0.17             1.18        1.00


In [21]:
one_time_age = (
    df_one_time["age"]
    .value_counts(normalize=True)
    .mul(100)
)

repeat_age = (
    df_repeat["age"]
    .value_counts(normalize=True)
    .mul(100)
)

age_comparison = pd.DataFrame({
    "one_time_percentage": one_time_age,
    "repeat_percentage": repeat_age
}).fillna(0)

age_order = [
    "[0-10)",
    "[10-20)",
    "[20-30)",
    "[30-40)",
    "[40-50)",
    "[50-60)",
    "[60-70)",
    "[70-80)",
    "[80-90)",
    "[90-100)"
]

age_comparison = age_comparison.reindex(age_order)

age_comparison["difference"] = (
        age_comparison["repeat_percentage"]
        - age_comparison["one_time_percentage"]
)

print(age_comparison.round(2))

          one_time_percentage  repeat_percentage  difference
age                                                         
[0-10)                   0.27               0.03       -0.23
[10-20)                  0.81               0.52       -0.29
[20-30)                  1.64               1.61       -0.03
[30-40)                  3.91               3.47       -0.44
[40-50)                  9.81               9.18       -0.62
[50-60)                 17.80              15.97       -1.83
[60-70)                 22.49              21.63       -0.86
[70-80)                 24.87              26.49        1.62
[80-90)                 15.66              18.34        2.69
[90-100)                 2.74               2.75        0.01


In [22]:
print("Top 20 primary diagnosis codes - One-time patients:")
print(
    df_one_time["diag_1"]
    .value_counts()
    .head(20)
)

print("\nTop 20 primary diagnosis codes - Repeat patients:")
print(
    df_repeat["diag_1"]
    .value_counts()
    .head(20)
)

Top 20 primary diagnosis codes - One-time patients:
diag_1
414      4126
428      2656
786      2381
410      2296
486      1781
427      1628
715      1583
434      1174
682      1101
780      1078
491       927
276       864
38        863
996       854
250.8     787
599       735
584       729
574       614
820       614
435       582
Name: count, dtype: int64

Top 20 primary diagnosis codes - Repeat patients:
diag_1
428      4206
414      2455
486      1727
786      1635
491      1348
410      1318
427      1138
996      1113
276      1025
682       941
780       941
250.8     893
599       860
434       854
38        825
584       791
250.6     768
V57       667
518       594
493       569
Name: count, dtype: int64


In [23]:
def categorize_diagnosis(code):
    if pd.isna(code) or code == "?":
        return "Unknown"

    code = str(code).strip()

    # V and E codes
    if code.startswith("V") or code.startswith("E"):
        return "Other"

    try:
        code_num = float(code)
    except ValueError:
        return "Other"

    if 390 <= code_num <= 459 or code_num == 785:
        return "Circulatory"

    elif 460 <= code_num <= 519 or code_num == 786:
        return "Respiratory"

    elif 520 <= code_num <= 579 or code_num == 787:
        return "Digestive"

    elif 250 <= code_num < 251:
        return "Diabetes"

    elif 800 <= code_num <= 999:
        return "Injury"

    elif 710 <= code_num <= 739:
        return "Musculoskeletal"

    elif 580 <= code_num <= 629 or code_num == 788:
        return "Genitourinary"

    elif 140 <= code_num <= 239:
        return "Neoplasms"

    else:
        return "Other"

df_one_time["diag_1_category"] = (
            df_one_time["diag_1"].apply(categorize_diagnosis)
        )

df_repeat["diag_1_category"] = (
    df_repeat["diag_1"].apply(categorize_diagnosis)
)

one_time_diag = (
    df_one_time["diag_1_category"]
    .value_counts(normalize=True)
    .mul(100)
)

repeat_diag = (
    df_repeat["diag_1_category"]
    .value_counts(normalize=True)
    .mul(100)
)

diagnosis_comparison = pd.DataFrame({
    "one_time_percentage": one_time_diag,
    "repeat_percentage": repeat_diag
}).fillna(0)

diagnosis_comparison["difference"] = (
        diagnosis_comparison["repeat_percentage"]
        - diagnosis_comparison["one_time_percentage"]
)

diagnosis_comparison = diagnosis_comparison.sort_values(
    "repeat_percentage",
    ascending=False
)

print(diagnosis_comparison.round(2))

                 one_time_percentage  repeat_percentage  difference
diag_1_category                                                    
Circulatory                    30.31              29.44       -0.87
Other                          17.29              18.52        1.23
Respiratory                    13.39              15.08        1.69
Diabetes                        7.81               9.53        1.71
Digestive                       9.21               9.43        0.23
Injury                          6.78               6.94        0.16
Genitourinary                   4.96               5.10        0.14
Musculoskeletal                 6.10               3.44       -2.65
Neoplasms                       4.13               2.49       -1.64
Unknown                         0.02               0.03        0.01


# Phase 2 — Data Understanding Summary

## Dataset Overview

- 101,766 hospital encounter records
- 50 columns
- 71,518 unique patients
- Each `encounter_id` is unique
- 0 fully duplicated rows
- Some patients have multiple encounters

## Readmission Distribution

- `NO`: 54,864 encounters (53.91%)
- `>30`: 35,545 encounters (34.93%)
- `<30`: 11,357 encounters (11.16%)

`<30` represents the outcome of interest for the later 30-day readmission analysis. The 11.16% is the percentage of encounter records in the raw dataset labeled `<30`, not a percentage of unique patients.

## Missing Data Findings

Missing information is represented in more than one way.

**Standard null/NaN:**

- `max_glu_serum`: 94.75%
- `A1Cresult`: 83.28%

**Question-mark placeholders observed:**

- `weight`: 96.86%
- `medical_specialty`: 49.08%
- `payer_code`: 39.56%
- `race`: 2.23%
- `diag_3`: 1.40%
- `diag_2`: 0.35%

The notebook summary reported 7 columns containing question-mark placeholders, while the previously printed detailed table showed 6. This is a data-quality discrepancy that should be investigated during Phase 3 rather than silently resolved.

## Repeat Patient Exploration

- 54,745 patients had exactly one encounter
- 16,773 patients had more than one encounter
- Repeat patients represent 23.45% of unique patients
- The 16,773 repeat patients account for 47,021 encounter records

**Encounter-frequency distribution:**

- 1 encounter: 54,745 patients (76.55%)
- 2 encounters: 10,434 patients (14.59%)
- 3 encounters: 3,328 patients (4.65%)
- 4 encounters: 1,421 patients (1.99%)
- 5+ encounters: 1,590 patients (2.22%)

## Exploratory Differences

The following comparisons are encounter-level exploratory findings and should not yet be interpreted as causal relationships or final patient-level conclusions.

**Prior inpatient utilization:**

- One-time patient encounters: 0.17 average
- Repeat-patient encounters: 1.18 average

**Other averages (one-time patient encounters vs repeat-patient encounters):**

- `time_in_hospital`: 4.22 vs 4.60
- `num_lab_procedures`: 43.38 vs 42.76
- `num_procedures`: 1.47 vs 1.19
- `num_medications`: 15.63 vs 16.48
- `number_outpatient`: 0.27 vs 0.49
- `number_emergency`: 0.09 vs 0.32

## Age Pattern

Repeat-patient encounters were somewhat more concentrated in older age groups. For example:

- Age 70–80: 24.87% one-time vs 26.49% repeat
- Age 80–90: 15.66% one-time vs 18.34% repeat

These observed differences do not establish that age causes repeat utilization.

## Diagnosis Pattern

The broader primary-diagnosis comparison showed:

- Diabetes: 7.81% one-time vs 9.53% repeat
- Respiratory: 13.39% one-time vs 15.08% repeat
- Circulatory: 30.31% one-time vs 29.44% repeat
- Musculoskeletal: 6.10% one-time vs 3.44% repeat
- Neoplasms: 4.13% one-time vs 2.49% repeat

Circulatory diagnoses were the largest category in both groups, while diabetes and respiratory diagnoses had somewhat greater representation among repeat-patient encounters.

## Important Limitations

- Multiple encounters from the same patient are not automatically 30-day readmissions.
- Repeat-patient analysis and 30-day readmission analysis are different concepts.
- Repeat-patient comparisons currently operate at the encounter level, so patients with many encounters contribute multiple observations.
- These findings describe associations in historical data and do not establish causation.
- The raw dataset has not yet been cleaned.
- Missing-value treatment decisions have not yet been made.
- Diagnosis grouping is currently exploratory and should be validated/documented before being used in final analysis.
- The primary project outcome remains 30-day readmission (`readmitted = <30`).

## Phase 3 — Data Cleaning & Transformation

Phase 3 will:

- standardize missing values,
- investigate the `?` column-count discrepancy,
- evaluate high-missingness columns,
- validate important categorical fields,
- create documented diagnosis categories,
- create the binary `readmit_30_flag`,
- and produce a cleaned dataset for SQL and downstream analysis.